<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet6%20(ANN)/Praktikum%206%20(JS6).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib

In [4]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

dpath = 'songs_with_attributes_and_lyrics.csv'

# Pakai engine='python' + on_bad_lines='skip' biar aman dari tanda petik lirik yang rusak
df = pd.read_csv(dpath, engine='python', on_bad_lines='skip')

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# 2. Hapus data yang ada nilai kosong (NaN)
df = df.dropna(subset=features)

X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
k = 10  # jumlah nearest neighbors

# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
# Cari cuma buat 1.000 lagu pertama biar gak 18 menit!
dist_exact, idx_exact = nn.kneighbors(X_scaled[:1000])
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
# Query 1.000 lagu pertama
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled[:1000]]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
# Query 1.000 lagu pertama
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled[:1000], k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])

# Posisi argumen dibetulkan: 100 dan faiss.METRIC_L2 langsung ditulis tanpa nama keyword
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10

dist_faiss, idx_faiss = index_faiss.search(X_scaled[:1000], k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

Exact NN done in 5.349 s
Annoy done in 21.233 s
HNSW done in 323.305 s
FAISS IVF done in 1.419 s

Top-5 neighbors for first song:
Exact NN: [     0 394553 764272 837727 749223]
Annoy:    [0, 764272, 749223, 625227, 833164]
HNSW:     [     0 394553 764272 837727 749223]
FAISS:    [     0 394553 764272 837727 749223]


Buat dan tuliskan analisa anda terhadap code diatas.

Jawab :

Saya memodifikasi kode dari jobsheet(X_scaled[:1000]), karena Kode asli pada jobsheet melakukan eksekusi pencarian nearest neighbors terhadap seluruh baris data (>500.000 lagu) secara bersamaan. Pencarian brute-force pada $500.000 \times 500.000$ pasangan titik serta looping manual Python pada Annoy membutuhkan waktu eksekusi lebih dari 30–60 menit. Dengan membatasi pencarian pada 1.000 lagu pertama, proses benchmark dapat selesai dalam hitungan detik/menit tanpa mengubah akurasi struktur index yang sudah dibangun dari seluruh dataset.

**Analisis :**

**A. Perbandingan Waktu Eksekus**

- FAISS IVF: 1.419 detik

- Exact NN: 5.349 detik

- Annoy: 21.233 detik

- HNSW: 323.305 detik

FAISS IVF menjadi tercepat dengan waktu 1,419 detik. Sebaliknya, HNSW membutuhkan waktu paling lama karena proses pembuatan index-nya cukup berat. Annoy juga membutuhkan waktu lebih lama karena pencarian dilakukan satu per satu menggunakan perulangan Python.

**B. Perbandingan Akurasi Top-5**

- Exact NN: [0, 394553, 764272, 837727, 749223]

- HNSW: Hasilnya sama dengan Exact NN, dengan kecocokan 100%.

- FAISS IVF: Hasilnya sama dengan Exact NN, dengan kecocokan 100%.

- Annoy: [0, 764272, 749223, 625227, 833164], dengan 3 dari 5 hasil yang cocok atau 60%.

HNSW dan FAISS IVF menghasilkan rekomendasi yang sama dengan Exact NN. Sementara itu, Annoy hanya menghasilkan 3 rekomendasi yang cocok dari 5 hasil teratas.

Kesimpulan

- StandardScaler: Digunakan untuk menyamakan skala setiap fitur audio agar perbedaan rentang nilai tidak terlalu memengaruhi perhitungan jarak.

- Kecepatan algoritma: FAISS IVF menjadi algoritma tercepat pada pengujian ini, sehingga cocok dipertimbangkan untuk pencarian lagu dalam dataset berukuran besar.

- Perbandingan algoritma: HNSW dan FAISS IVF memiliki kecocokan hasil 100% terhadap Exact NN pada pengujian Top-5. Annoy memiliki kecocokan 60%, sedangkan waktu eksekusi HNSW paling lama.